# UNIFORM TELEPORTATION 

- Implementation of the map equation with the basic uniform-teleportation scheme (matches Laura's). The random walker either follows an out-edge (weighted by edge weight) or, with probability τ, teleports to any node in the network with equal probability 1/N. Teleportation steps are "recorded"; they count as part of the encoded trajectory, so they contribute to the exit probability of each community via the term τ·(1 - n_α/N)·p_α.

- It's used here as a sanity check that the core formula is implemented correctly before moving on to the smart unrecorded scheme that the paper actually recommends (the non-uniform).

## Validation 

- Compare against igraph's reported codelength (for the directed obvs).
- Also compare against Laura's reported numbers.

- combos: {weighted, unweighted} × {directed, undirected}

In [7]:
import igraph as ig
import numpy as np
import warnings

In [8]:
# apparently i need this
def safe_xlogx(x):
    return np.where(x > 0, x * np.log2(x), 0.0)

In [9]:
def pagerank_uniform(M, tau=0.15, tol=1e-15, maxiter=int(1e6)):
    """Standard PageRank with uniform teleportation (d = 1/N for all nodes)."""
    N = M.shape[0]
    row_sums = M.sum(axis=1)
    dangling = (row_sums == 0)
    row_sums_safe = np.where(dangling, 1, row_sums)
    M_norm = M / row_sums_safe[:, None]

    d = np.ones(N) / N   # uniform teleportation

    p = np.ones(N) / N
    for _ in range(int(maxiter)):
        dangling_sum = p[dangling].sum()
        p_new = (1 - tau) * (p @ M_norm + dangling_sum * d) + tau * d
        if np.linalg.norm(p_new - p) < tol:
            return p_new
        p = p_new
    warnings.warn(f"PageRank did not converge after {maxiter} iterations.")
    return p

In [10]:
def compute_exit_flow(g, communities, p):
    """Sum of p[src] * weight / out_strength[src] over edges leaving each community."""
    communities = np.array(communities)
    out_strength = np.array(g.strength(mode="out", weights="weight" if g.is_weighted() else None), dtype=float)
    weights = np.array(g.es["weight"] if g.is_weighted() else [1.0] * g.ecount())
    edges = np.array(g.get_edgelist(), dtype=int)

    src, trg = edges[:, 0], edges[:, 1]
    src_com, trg_com = communities[src], communities[trg]
    betw = src_com != trg_com

    out_str_safe = np.where(out_strength > 0, out_strength, 1.0)
    flow = p[src] * weights / out_str_safe[src]

    exit_flow = np.zeros(max(communities) + 1)
    np.add.at(exit_flow, src_com[betw], flow[betw])
    return exit_flow

In [11]:
def compute_exit_weights(g, communities):
    """For undirected graphs: total weight of edges crossing community boundaries (counted from both sides)."""
    communities = np.array(communities)
    weights = np.array(g.es["weight"] if g.is_weighted() else [1.0] * g.ecount())
    edges = np.array(g.get_edgelist(), dtype=int)

    src_com = communities[edges[:, 0]]
    trg_com = communities[edges[:, 1]]
    betw = src_com != trg_com

    exit_weights = np.zeros(max(communities) + 1)
    np.add.at(exit_weights, src_com[betw], weights[betw])
    np.add.at(exit_weights, trg_com[betw], weights[betw])
    return exit_weights

In [12]:
def compute_description_length(g, communities, tau=0.15):
    """Map equation L for directed and undirected graphs, using uniform recorded teleportation."""
    communities = np.array(communities)
    num_comms = max(communities) + 1
    N = g.vcount()

    if g.is_directed():
        adj = np.array(g.get_adjacency(attribute="weight" if g.is_weighted() else None).data, dtype=float)
        p = pagerank_uniform(adj, tau=tau)

        p_mod = np.zeros(num_comms)
        np.add.at(p_mod, communities, p)

        exit_flow = compute_exit_flow(g, communities, p)

        # recorded uniform teleportation: includes the teleportation term
        n_mod = np.bincount(communities, minlength=num_comms).astype(float)
        q_mod = tau * (1 - n_mod / N) * p_mod + (1 - tau) * exit_flow
    else:
        weights = np.array(g.es["weight"] if g.is_weighted() else [1.0] * g.ecount())
        total_weight_x2 = 2 * np.sum(weights)
        strength = np.array(g.strength(weights="weight" if g.is_weighted() else None), dtype=float)
        p = strength / total_weight_x2

        p_mod = np.zeros(num_comms)
        np.add.at(p_mod, communities, p)

        exit_weights = compute_exit_weights(g, communities)
        q_mod = exit_weights / total_weight_x2

    q_sum = np.sum(q_mod)
    p_loop = p_mod + q_mod

    L = safe_xlogx(q_sum) - 2 * np.sum(safe_xlogx(q_mod)) \
        - np.sum(safe_xlogx(p)) + np.sum(safe_xlogx(p_loop))
    return L

In [21]:
# unweighted + undirected
g = ig.Graph.Read_GraphML("C:/Users/savin/ITI/unweighted_undirected.graphml")
communities = g.community_infomap()
print("--- unweighted + undirected ---")
print(f"Custom L: {compute_description_length(g, communities.membership):.6f}")
print(f"igraph L: {communities.codelength:.6f}")

# weighted + undirected
g = ig.Graph.Read_GraphML("C:/Users/savin/ITI/weighted_undirected.graphml")
communities = g.community_infomap(edge_weights="weight")
print("\n--- weighted + undirected ---")
print(f"Custom L: {compute_description_length(g, communities.membership):.6f}")
print(f"igraph L: {communities.codelength:.6f}")

# unweighted + directed
g = ig.Graph.Read_GraphML("C:/Users/savin/ITI/unweighted_directed.graphml")
communities = g.community_infomap()
print("\n--- unweighted + directed ---")
print(f"Custom L: {compute_description_length(g, communities.membership):.6f}   (Lauras: 3.754)")
print(f"igraph L: {communities.codelength:.6f}")

# weighted + directed
g = ig.Graph.Read_GraphML("C:/Users/savin/ITI/weighted_directed.graphml")
communities = g.community_infomap(edge_weights="weight")
communities_no_weights = g.community_infomap()   
print("\n--- weighted + directed ---")
print(f"Custom L: {compute_description_length(g, communities.membership):.6f} but eq. to Laura's {compute_description_length(g, communities_no_weights.membership):.6f} (Lauras: 3.485)")
print(f"igraph L: {communities.codelength:.6f}")

--- unweighted + undirected ---
Custom L: 3.401583
igraph L: 3.401583

--- weighted + undirected ---
Custom L: 3.336712
igraph L: 3.336712

--- unweighted + directed ---
Custom L: 3.754023   (Lauras: 3.754)
igraph L: 3.515830

--- weighted + directed ---
Custom L: 3.495510 but eq. to Laura's 3.484882 (Lauras: 3.485)
igraph L: 3.392047


## NOTE

Laura found 3.485 (lower) for the weighted directed case, but the value should be 3.495 because probabky forgot to pass edge_weights="weight" to community_infomap so igraph clustered the graph as unweighted and evaluated L with weights on that partition.
Just for the sake of it i tried the same and reproduced it and its the same :'))))